# Chapter 9 &mdash; Checking the Conversion by Round-Tripping to Minimal DFA

**Concept 5 of the Chapter 9 decomposition:** *Checking the Conversion by Round-Tripping to Minimal DFA*

Convert the produced RE back to a minimal DFA and check `iso_dfa` against the original.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter9-NFA2RE/Concept-Round-Trip-Check/Concept-Round-Trip-Check.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.Def_NFA2RE     import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateDFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateDFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


The conversion produces a large, unreadable RE. How do you know it is right?

**Round-trip it.** Convert the RE back with `re2nfa`, determinize, minimize, and
compare with the minimized original using `iso_dfa`. By Myhill&ndash;Nerode a `True`
answer means the languages are **identical** &mdash; not merely similar.

$$N \xrightarrow{\ \text{NFA2RE}\ } R \xrightarrow{\ \text{RE2NFA}\ } N' \xrightarrow{\ \text{det, min}\ } D' \;\cong\; \text{min}(N)$$

This is the standard way to test *any* language-preserving transformation, and it
costs four function calls.

## 2. Definitions

### The round trip, as one function

In [ ]:
def round_trip(N):
    _, _, r = del_gnfa_states(mk_gnfa(N))
    return r, min_dfa(nfa2dfa(N)), min_dfa(nfa2dfa(re2nfa(r)))

### A batch of machines to try it on

In [ ]:
MACHINES = {
 'even 0s'      : '''NFA
IF : 0 -> A
IF : 1 -> IF
A  : 0 -> IF
A  : 1 -> A
''',
 'ends in 01'   : '''NFA
I : 0 | 1 -> I
I : 0 -> A
A : 1 -> F
''',
 'contains 11'  : '''NFA
I : 0 | 1 -> I
I : 1 -> A
A : 1 -> F
F : 0 | 1 -> F
''',
 'third-last 1' : '''NFA
I : 0 | 1 -> I
I : 1 -> A
A : 0 | 1 -> B
B : 0 | 1 -> F
''',
}

<!-- nav-strip -->

---

&larr;&nbsp;[Ch9&nbsp;4.&nbsp;A Non-Trivial Conversion, Step by Step](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter9-NFA2RE/Concept-Non-Trivial-Conversion/Concept-Non-Trivial-Conversion.ipynb) &nbsp;&middot;&nbsp; [**Chapter 9** index](https://github.com/ganeshutah/Jove/blob/master/Chapter9-NFA2RE/README.md) &nbsp;&middot;&nbsp; [Ch9&nbsp;6.&nbsp;DFA, NFA, and RE Are Equally Powerful](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter9-NFA2RE/Concept-Three-Models-Equally-Powerful/Concept-Three-Models-Equally-Powerful.ipynb)&nbsp;&rarr;

---

## 3. Tests

Every machine round-trips to an isomorphic minimal DFA.

In [ ]:
results = {}
for name, src in MACHINES.items():
    N = md2mc(src)
    r, D0, D1 = round_trip(N)
    results[name] = (len(r), len(D0["Q"]), len(D1["Q"]), iso_dfa(D0, D1))
    print("%-15s RE len %4d  min |Q| %2d vs %2d  iso %s" % ((name,) + results[name]))
    assert iso_dfa(D0, D1)

String-level cross-check, for good measure.

In [ ]:
from itertools import product
strs = [''.join(p) for k in range(10) for p in product('01', repeat=k)]
for name, src in MACHINES.items():
    N = md2mc(src)
    r, _, D1 = round_trip(N)
    assert all(accepts_nfa(N, s) == accepts_dfa(D1, s) for s in strs)
    print("%-15s agrees on all %d strings" % (name, len(strs)))

A **broken** transformation is caught immediately.

In [ ]:
N = md2mc(MACHINES['ends in 01'])
r, D0, _ = round_trip(N)
broken = r + "0"                       # append a stray symbol to the RE
Db = min_dfa(nfa2dfa(re2nfa(broken)))
print("tampered RE round-trips to an isomorphic machine? ", iso_dfa(D0, Db))
assert not iso_dfa(D0, Db)
langeq_dfa(D0, Db, gen_counterex=True)

And the check is cheap: four calls, whatever the machine size.

In [ ]:
print("round_trip = del_gnfa_states + re2nfa + nfa2dfa + min_dfa")
print("cost is dominated by nfa2dfa, which is exponential in the worst case --")
print("but on the machines you actually write, it is instant.")

## 4. Animation

One of the round-tripped machines.

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(min_dfa(nfa2dfa(md2mc(MACHINES['contains 11']))), FuseEdges=True)

## 5. Exercises


1. Round-trip a machine **twice**. Does the RE stabilise?
2. Why is `iso_dfa` the right comparison here rather than string testing?
3. What does the counterexample walk print for the tampered RE?

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter9-NFA2RE/Concept-Round-Trip-Check')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')